In [39]:
import pandas as pd
import numpy as np

df = pd.read_parquet(
    r"../data/interim/preprocessed_chunks/chunk_0001.parquet"
)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nDtypes:")
print(df.dtypes.head(20))

Shape: (5739000, 22)

Columns:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'day', 'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price']

Dtypes:
id              category
item_id         category
dept_id         category
cat_id          category
store_id        category
state_id        category
day               object
sales              int16
date              object
wm_yr_wk           int16
weekday         category
wday                int8
month               int8
year               int16
event_name_1    category
event_type_1    category
event_name_2    category
event_type_2    category
snap_CA             int8
snap_TX             int8
dtype: object


In [40]:
df["date"] = pd.to_datetime(df["date"])

df = df.sort_values(
    ["id", "date"]
).reset_index(drop=True)

print(df[["id", "date", "sales"]].head())
print(df[["id", "date", "sales"]].info(memory_usage="deep"))

                            id       date  sales
0  FOODS_1_001_CA_1_validation 2011-01-29      3
1  FOODS_1_001_CA_1_validation 2011-01-30      0
2  FOODS_1_001_CA_1_validation 2011-01-31      0
3  FOODS_1_001_CA_1_validation 2011-02-01      1
4  FOODS_1_001_CA_1_validation 2011-02-02      4
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5739000 entries, 0 to 5738999
Data columns (total 3 columns):
 #   Column  Dtype         
---  ------  -----         
 0   id      category      
 1   date    datetime64[ns]
 2   sales   int16         
dtypes: category(1), datetime64[ns](1), int16(1)
memory usage: 66.0 MB
None


In [41]:
lag_days = [1, 7, 14, 28]

for lag in lag_days:
    df[f"lag_{lag}"] = (
        df.groupby("id")["sales"]
          .shift(lag)
          .astype("float32")
    )

C:\Users\mohme\AppData\Local\Temp\ipykernel_28352\1757443228.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("id")["sales"]
C:\Users\mohme\AppData\Local\Temp\ipykernel_28352\1757443228.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("id")["sales"]
C:\Users\mohme\AppData\Local\Temp\ipykernel_28352\1757443228.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("id")["sales"]
C:\User

In [42]:
df[
    [
        "id",
        "date",
        "sales",
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_28"
    ]
].head(35)

,id,date,sales,lag_1,lag_7,lag_14,lag_28
0,FOODS_1_001_CA_1_validation,2011-01-29,3,NaN,NaN,NaN,NaN
1,FOODS_1_001_CA_1_validation,2011-01-30,0,3.0,NaN,NaN,NaN
2,FOODS_1_001_CA_1_validation,2011-01-31,0,0.0,NaN,NaN,NaN
3,FOODS_1_001_CA_1_validation,2011-02-01,1,0.0,NaN,NaN,NaN
4,FOODS_1_001_CA_1_validation,2011-02-02,4,1.0,NaN,NaN,NaN
5,FOODS_1_001_CA_1_validation,2011-02-03,2,4.0,NaN,NaN,NaN
6,FOODS_1_001_CA_1_validation,2011-02-04,0,2.0,NaN,NaN,NaN
7,FOODS_1_001_CA_1_validation,2011-02-05,2,0.0,3.0,NaN,NaN
8,FOODS_1_001_CA_1_validation,2011-02-06,0,2.0,0.0,NaN,NaN
9,FOODS_1_001_CA_1_validation,2011-02-07,0,0.0,0.0,NaN,NaN


In [43]:
df[
    ["lag_1","lag_7","lag_14","lag_28"]
].isnull().sum()


lag_1      3000
lag_7     21000
lag_14    42000
lag_28    84000
dtype: int64

In [44]:
windows = [7, 14, 28]

for window in windows:
    df[f"rolling_mean_{window}"] = (
        df.groupby("id")["sales"]
          .transform(
              lambda x: x.shift(1).rolling(window).mean()
          )
          .astype("float32")
    )

    df[f"rolling_std_{window}"] = (
        df.groupby("id")["sales"]
          .transform(
              lambda x: x.shift(1).rolling(window).std()
          )
          .astype("float32")
    )

C:\Users\mohme\AppData\Local\Temp\ipykernel_28352\3936617268.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("id")["sales"]
C:\Users\mohme\AppData\Local\Temp\ipykernel_28352\3936617268.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("id")["sales"]
C:\Users\mohme\AppData\Local\Temp\ipykernel_28352\3936617268.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("id")["sales"]
C:\Use

In [45]:
rolling_cols = [
    c for c in df.columns
    if "rolling" in c
]

print(df[rolling_cols].isnull().sum())

rolling_mean_7     21000
rolling_std_7      21000
rolling_mean_14    42000
rolling_std_14     42000
rolling_mean_28    84000
rolling_std_28     84000
dtype: int64


In [46]:
df["is_weekend"] = (df["wday"] >= 6).astype("int8")

In [47]:
df["price_lag_1"] = (
    df.groupby("id")["sell_price"]
      .shift(1)
      .astype("float32")
)

C:\Users\mohme\AppData\Local\Temp\ipykernel_28352\515424920.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("id")["sell_price"]


In [48]:
df["price_diff_1"] = (
    df["sell_price"] - df["price_lag_1"]
).astype("float32")

In [49]:
df["price_pct_change"] = (
    df.groupby("id")["sell_price"]
      .pct_change()
      .astype("float32")
)

C:\Users\mohme\AppData\Local\Temp\ipykernel_28352\664977872.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("id")["sell_price"]
C:\Users\mohme\AppData\Local\Temp\ipykernel_28352\664977872.py:3: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change()


In [50]:
df["price_max"] = (
    df.groupby("id")["sell_price"]
      .transform("max")
      .astype("float32")
)

C:\Users\mohme\AppData\Local\Temp\ipykernel_28352\4293512432.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("id")["sell_price"]


In [51]:
df["price_relative_to_max"] = (
    df["sell_price"] / df["price_max"]
).astype("float32")

In [52]:
new_price_cols = [
    "price_diff_1",
    "price_pct_change",
    "price_relative_to_max"
]

print(df[new_price_cols].describe())

       price_diff_1  price_pct_change  price_relative_to_max
count  4.624386e+06      4.624386e+06           4.627386e+06
mean   6.293161e-05      1.095920e-04           9.606075e-01
std    3.172028e-02      1.178399e-01           6.307563e-02
min   -1.597000e+01     -9.959514e-01           3.787879e-03
25%    0.000000e+00      0.000000e+00           9.328859e-01
50%    0.000000e+00      0.000000e+00           1.000000e+00
75%    0.000000e+00      0.000000e+00           1.000000e+00
max    1.597000e+01      2.460000e+02           1.000000e+00


In [53]:
df.isnull().sum().sort_values(ascending=False).head(20)

price_pct_change         1114614
price_diff_1             1114614
price_lag_1              1114614
price_relative_to_max    1111614
sell_price               1111614
rolling_std_28             84000
rolling_mean_28            84000
lag_28                     84000
rolling_mean_14            42000
lag_14                     42000
rolling_std_14             42000
rolling_std_7              21000
rolling_mean_7             21000
lag_7                      21000
lag_1                       3000
item_id                        0
is_weekend                     0
price_max                      0
snap_WI                        0
id                             0
dtype: int64

In [54]:
df["sell_price"] = (
    df.groupby("id")["sell_price"]
      .ffill()
      .bfill()
)

C:\Users\mohme\AppData\Local\Temp\ipykernel_28352\2609236432.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("id")["sell_price"]


In [55]:
df["price_lag_1"] = (
    df.groupby("id")["sell_price"]
      .shift(1)
      .astype("float32")
)

df["price_diff_1"] = (
    df["sell_price"] - df["price_lag_1"]
).astype("float32")

df["price_pct_change"] = (
    df.groupby("id")["sell_price"]
      .pct_change()
      .astype("float32")
)

df["price_max"] = (
    df.groupby("id")["sell_price"]
      .transform("max")
      .astype("float32")
)

df["price_relative_to_max"] = (
    df["sell_price"] / df["price_max"]
).astype("float32")

C:\Users\mohme\AppData\Local\Temp\ipykernel_28352\2962595709.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("id")["sell_price"]
C:\Users\mohme\AppData\Local\Temp\ipykernel_28352\2962595709.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("id")["sell_price"]
C:\Users\mohme\AppData\Local\Temp\ipykernel_28352\2962595709.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("id")["sel

In [56]:
critical_cols = [
    "lag_28",
    "rolling_mean_28",
    "rolling_std_28",
    "price_lag_1",
    "price_pct_change",
    "price_relative_to_max"
]

df = df.dropna(subset=critical_cols)

In [57]:
df.isnull().sum().sort_values(ascending=False).head(20)

id                  0
rolling_mean_14     0
sell_price          0
lag_1               0
lag_7               0
lag_14              0
lag_28              0
rolling_mean_7      0
rolling_std_7       0
rolling_std_14      0
item_id             0
rolling_mean_28     0
rolling_std_28      0
is_weekend          0
price_lag_1         0
price_diff_1        0
price_pct_change    0
price_max           0
snap_WI             0
snap_TX             0
dtype: int64

In [59]:
import os
import glob
import pandas as pd
import numpy as np

input_dir = r"../data/interim/preprocessed_chunks"
output_dir = r"../data/processed/feature_chunks"

os.makedirs(output_dir, exist_ok=True)

files = sorted(glob.glob(os.path.join(input_dir, "*.parquet")))

for file in files:

    print(f"Processing: {os.path.basename(file)}")

    # ================= Load =================
    df = pd.read_parquet(file)

    # ================= Prepare =================
    df["date"] = pd.to_datetime(df["date"])

    df = (
        df.sort_values(["id", "date"])
          .reset_index(drop=True)
    )

    # ================= Lag Features =================
    for lag in [1, 7, 14, 28]:
        df[f"lag_{lag}"] = (
            df.groupby("id")["sales"]
              .shift(lag)
              .astype("float32")
        )

    # ================= Rolling Features =================
    for window in [7, 14, 28]:

        df[f"rolling_mean_{window}"] = (
            df.groupby("id")["sales"]
              .transform(
                  lambda x: x.shift(1).rolling(window).mean()
              )
              .astype("float32")
        )

        df[f"rolling_std_{window}"] = (
            df.groupby("id")["sales"]
              .transform(
                  lambda x: x.shift(1).rolling(window).std()
              )
              .astype("float32")
        )

    # ================= Calendar Feature =================
    df["is_weekend"] = (df["wday"] >= 6).astype("int8")

    # ================= Price Handling =================
    df["sell_price"] = (
        df.groupby("id")["sell_price"]
          .ffill()
          .bfill()
    )

    # ================= Price Features =================
    df["price_lag_1"] = (
        df.groupby("id")["sell_price"]
          .shift(1)
          .astype("float32")
    )

    df["price_diff_1"] = (
        df["sell_price"] - df["price_lag_1"]
    ).astype("float32")

    df["price_pct_change"] = (
        df.groupby("id")["sell_price"]
          .pct_change()
          .astype("float32")
    )

    df["price_max"] = (
        df.groupby("id")["sell_price"]
          .transform("max")
          .astype("float32")
    )

    df["price_relative_to_max"] = (
        df["sell_price"] / df["price_max"]
    ).astype("float32")

    # ================= Remove NaN =================
    critical_cols = [
        "lag_28",
        "rolling_mean_28",
        "rolling_std_28",
        "price_lag_1",
        "price_pct_change",
        "price_relative_to_max"
    ]

    df = df.dropna(subset=critical_cols)

    # ================= Save =================
    output_path = os.path.join(
        output_dir,
        f"features_{os.path.basename(file)}"
    )

    df.to_parquet(output_path, index=False)

    print(f"Saved -> {output_path}")
    print("-" * 50)

print("Feature Engineering completed successfully.")

Processing: chunk_0001.parquet


C:\Users\mohme\AppData\Local\Temp\ipykernel_28352\2627518339.py:31: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("id")["sales"]
C:\Users\mohme\AppData\Local\Temp\ipykernel_28352\2627518339.py:31: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("id")["sales"]
C:\Users\mohme\AppData\Local\Temp\ipykernel_28352\2627518339.py:31: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("id")["sales"]
C:\U

Saved -> D:\gearofproml\day5\feature_chunks\features_chunk_0001.parquet
--------------------------------------------------
Processing: chunk_0002.parquet


C:\Users\mohme\AppData\Local\Temp\ipykernel_28352\2627518339.py:31: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("id")["sales"]
C:\Users\mohme\AppData\Local\Temp\ipykernel_28352\2627518339.py:31: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("id")["sales"]
C:\Users\mohme\AppData\Local\Temp\ipykernel_28352\2627518339.py:31: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("id")["sales"]
C:\U

Saved -> D:\gearofproml\day5\feature_chunks\features_chunk_0002.parquet
--------------------------------------------------
Processing: chunk_0003.parquet


...

Saved -> D:\gearofproml\day5\feature_chunks\features_chunk_0009.parquet
--------------------------------------------------
Processing: chunk_0010.parquet


C:\Users\mohme\AppData\Local\Temp\ipykernel_28352\2627518339.py:31: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("id")["sales"]
C:\Users\mohme\AppData\Local\Temp\ipykernel_28352\2627518339.py:31: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("id")["sales"]
C:\Users\mohme\AppData\Local\Temp\ipykernel_28352\2627518339.py:31: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("id")["sales"]
C:\U

Saved -> D:\gearofproml\day5\feature_chunks\features_chunk_0010.parquet
--------------------------------------------------
Processing: chunk_0011.parquet


C:\Users\mohme\AppData\Local\Temp\ipykernel_28352\2627518339.py:31: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("id")["sales"]
C:\Users\mohme\AppData\Local\Temp\ipykernel_28352\2627518339.py:31: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("id")["sales"]
C:\Users\mohme\AppData\Local\Temp\ipykernel_28352\2627518339.py:31: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("id")["sales"]
C:\U

Saved -> D:\gearofproml\day5\feature_chunks\features_chunk_0011.parquet
--------------------------------------------------
Feature Engineering completed successfully.


In [60]:
df_test = pd.read_parquet(
    r"../data/processed/feature_chunks/features_chunk_0001.parquet"
)

In [61]:
print(df_test.shape)

(5655000, 38)


In [62]:
expected_features = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_std_7",
    "rolling_mean_14",
    "rolling_std_14",
    "rolling_mean_28",
    "rolling_std_28",
    "price_lag_1",
    "price_diff_1",
    "price_pct_change",
    "price_max",
    "price_relative_to_max",
    "is_weekend"
]

missing_features = [
    col for col in expected_features
    if col not in df_test.columns
]

print("Missing Features:", missing_features)

Missing Features: []


In [63]:
print(
    df_test.isnull().sum()
           .sort_values(ascending=False)
           .head(20)
)

id                  0
rolling_mean_14     0
sell_price          0
lag_1               0
lag_7               0
lag_14              0
lag_28              0
rolling_mean_7      0
rolling_std_7       0
rolling_std_14      0
item_id             0
rolling_mean_28     0
rolling_std_28      0
is_weekend          0
price_lag_1         0
price_diff_1        0
price_pct_change    0
price_max           0
snap_WI             0
snap_TX             0
dtype: int64


In [64]:
sample_id = df_test["id"].iloc[0]

cols = [
    "date",
    "sales",
    "lag_1",
    "lag_7",
    "rolling_mean_7",
    "sell_price",
    "price_lag_1",
    "price_diff_1"
]

print(
    df_test[df_test["id"] == sample_id][cols]
    .head(15)
)

         date  sales  lag_1  lag_7  rolling_mean_7  sell_price  price_lag_1  \
0  2011-02-26      2    4.0    1.0        1.857143         2.0          2.0   
1  2011-02-27      2    2.0    2.0        2.000000         2.0          2.0   
2  2011-02-28      0    2.0    0.0        2.000000         2.0          2.0   
3  2011-03-01      2    0.0    2.0        2.000000         2.0          2.0   
4  2011-03-02      1    2.0    2.0        2.000000         2.0          2.0   
5  2011-03-03      7    1.0    2.0        1.857143         2.0          2.0   
6  2011-03-04      1    7.0    4.0        2.571429         2.0          2.0   
7  2011-03-05      2    1.0    2.0        2.142857         2.0          2.0   
8  2011-03-06      3    2.0    2.0        2.142857         2.0          2.0   
9  2011-03-07      0    3.0    0.0        2.285714         2.0          2.0   
10 2011-03-08      1    0.0    2.0        2.285714         2.0          2.0   
11 2011-03-09      2    1.0    1.0        2.142857  